# Spine XR — Project 3 FINALE (Traditional vs ProGAN vs Diffusion + jury filtering)

Colab Pro+ (A100). ROI-128 patches, DenseNet-121, 3-fold CV + held-out test, online aug.
Generative: MONAI DDPM + ProGAN, with **confidence-based rejection sampling** (3-fold baseline ensemble jury) to keep only realistic synthetic patches.

**Dürüstlük:** final değerlendirme gerçek held-out test'te → filtreleme yanlılığı test kazancını taklit edemez.

## 1. Bootstrap

In [ ]:
# 1. Drive Mount
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
from pathlib import Path

# 2. Çalışma Alanını Yerel SSD'de Ayarla (A100'ün maksimum hızı için)
LOCAL_ROOT = Path('/content/spine-xr-augmentation-study')
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(LOCAL_ROOT)

# 3. Kodları Drive'dan Yerele Kopyala
DRIVE_REPO_PATH = Path('/content/drive/MyDrive/spine-xr-augmentation-study/spine-xr-augmentation-study')
!cp -r {DRIVE_REPO_PATH}/* .

# 4. Dataset'i SSD'ye Çek ve Aç (Dataset Drive'da .rar olarak durmalı)
# Klasör Adı "dataset" olmalı — configs/base.yaml relatif `dataset/...` yolları kullanır.
DRIVE_DATASET_PATH = Path('/content/drive/MyDrive/spine-xr-augmentation-study/dataset.rar')
!unrar x -o+ {DRIVE_DATASET_PATH} {LOCAL_ROOT}/

# 5. Çıktıların (Outputs) Kaybolmaması İçin Drive'a Bağla
DRIVE_OUTPUTS = Path('/content/drive/MyDrive/spine-xr-augmentation-study/outputs')
DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)

if os.path.exists('outputs') and not os.path.islink('outputs'):
    shutil.rmtree('outputs')
elif os.path.islink('outputs'):
    os.remove('outputs')
os.symlink(DRIVE_OUTPUTS, 'outputs')

print(f"Çalışma dizini (SSD): {os.getcwd()}")
!ls -l

In [ ]:
!pip install -q -r requirements.txt
!python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"
!python -c "import monai; print('monai', monai.__version__)"
!python -c "import monai; print('monai', monai.__version__)"

## 0. ROADMAP / ÇALIŞTIRMA SIRASI  ⚑ ÖNCE BUNU OKU

Notebook iki boru hattı içerir:
- **PATCH-LEVEL (eski, §3–§8):** lokalizasyon-verili üst sınır + **jüri** + **sentetik üretimi**. Bir kısmı GEREKLİ.
- **PATIENT-LEVEL (§9, HEADLINE):** hocanın istediği gerçek test (full image, box yok).

### Patient-level sonuç için GEREKLİ olanlar (sırayla)
1. `§1` Bootstrap → `§2` Audit/Splits/ROI/folds
2. `§3` CV Baseline — **bu modeller sentetik filtreleme JÜRİSİ'dir** (gerekli)
3. `§5` Diffusion: train → 5.sel → 5.pool → 5.filter  → `07f_diffusion_filtered`
4. `§6` ProGAN: train → 6.pool → 6.filter → `07f_progan_filtered`
5. `§9` Patient-level: 9.1 pencereler → 9.2 virtual patients → 9.3–9.6 eğit → 9.7 calibrate+test → 9.8 tablo
6. `§8` Rapor (patient-level dahil)

### OPSİYONEL (patch-level ek / üst sınır — patient-level için GEREKMEZ)
`§4` CV-traditional, `§5.cv`, `§6.cv`, `§7` patch karşılaştırma.

### ⏳ Şu an neredesin (5.cv'ye kadar çalıştırdın)
→ `07f_diffusion_filtered` HAZIR. **Sıradaki:** `§6` (ProGAN: 6.0 smoke → 6.1–6.6 → 6.pool → 6.filter) → sonra `§9` → `§8`.
`§4`, `§5.cv`, `§6.cv`, `§7`'yi atlayabilirsin.

Detaylı adım-adım anlatım: `docs/PROJECT_FLOW.md`.

## 2. Audit → Splits → ROI patches → CV folds [GEREKLİ]

In [ ]:
!python scripts/01_audit.py --config configs/base.yaml
!python scripts/02_data_splitter.py --config configs/base.yaml --cases configs/cases.yaml
!python scripts/02b_build_roi_patches.py --config configs/base.yaml --cases configs/cases.yaml
!python scripts/02c_make_roi_folds.py --config configs/base.yaml --cases configs/cases.yaml
!cat outputs/02b_roi/summary.md

## 3. CV Baseline — DenseNet-121 (no aug) — JÜRİ de buradan gelir [GEREKLİ — JÜRİ]

In [ ]:
!python scripts/03_train_cv.py --config configs/base.yaml --cases configs/cases.yaml \
    --classifier configs/classifier.yaml --train-transform baseline_no_aug --out-tag 03cv_baseline
!cat outputs/03cv_baseline/cv_summary.md

## 4. CV Traditional — online aug + balanced sampler [OPSİYONEL — patch-level ek]

In [ ]:
!python scripts/03_train_cv.py --config configs/base.yaml --cases configs/cases.yaml \
    --classifier configs/classifier.yaml --train-transform roi_train_online --balanced-sampler --out-tag 04cv_traditional
!cat outputs/04cv_traditional/cv_summary.md

## 5. Diffusion arm — MONAI DDPM + FID seçim + jüri filtre [GEREKLİ — sentetik üretimi]

### 5.0 Smoke (Other lesions, 300 iter)

In [ ]:
!python scripts/05d_train_diffusion.py --config configs/base.yaml --diffusion configs/diffusion.yaml \
    --classes-filter "Other lesions" --iterations 300 --out-tag 05d_smoke

### 5.1 DDPM train — Disc space narrowing

In [ ]:
!python scripts/05d_train_diffusion.py --config configs/base.yaml --diffusion configs/diffusion.yaml \
    --classes-filter "Disc space narrowing" --out-tag 05d_diffusion

### 5.2 DDPM train — Vertebral collapse

In [ ]:
!python scripts/05d_train_diffusion.py --config configs/base.yaml --diffusion configs/diffusion.yaml \
    --classes-filter "Vertebral collapse" --out-tag 05d_diffusion

### 5.3 DDPM train — Foraminal stenosis

In [ ]:
!python scripts/05d_train_diffusion.py --config configs/base.yaml --diffusion configs/diffusion.yaml \
    --classes-filter "Foraminal stenosis" --out-tag 05d_diffusion

### 5.4 DDPM train — Spondylolysthesis

In [ ]:
!python scripts/05d_train_diffusion.py --config configs/base.yaml --diffusion configs/diffusion.yaml \
    --classes-filter "Spondylolysthesis" --out-tag 05d_diffusion

### 5.5 DDPM train — Surgical implant

In [ ]:
!python scripts/05d_train_diffusion.py --config configs/base.yaml --diffusion configs/diffusion.yaml \
    --classes-filter "Surgical implant" --out-tag 05d_diffusion

### 5.6 DDPM train — Other lesions

In [ ]:
!python scripts/05d_train_diffusion.py --config configs/base.yaml --diffusion configs/diffusion.yaml \
    --classes-filter "Other lesions" --out-tag 05d_diffusion

### 5.sel FID ile en iyi checkpoint seçimi (Q1) — 8k değil, en düşük FID

In [ ]:
!python scripts/06d_select_diffusion_ckpt.py --config configs/base.yaml --diffusion configs/diffusion.yaml \
    --fid-samples 512 --out-tag 06d_ckpt_select
import json, glob
for f in sorted(glob.glob('outputs/06d_ckpt_select/*.json')):
    s=json.load(open(f)); print(s['class_slug'],'best iter',s['best_iter'],'FID',round(s['best_fid'],2))

### 5.pool Büyük havuz (5000) üret — seçilen checkpoint'ten

In [ ]:
import json, subprocess
from pathlib import Path
dmap=[('Disc space narrowing','case_1','disc_space_narrowing'),('Vertebral collapse','case_1','vertebral_collapse'),('Foraminal stenosis','case_2','foraminal_stenosis'),('Spondylolysthesis','case_2','spondylolysthesis'),('Surgical implant','case_3','surgical_implant'),('Other lesions','case_4','other_lesions')]
for nm,case,slug in dmap:
    sel=json.loads(Path(f'outputs/06d_ckpt_select/{slug}.json').read_text())
    print(slug,'<-',sel['best_checkpoint'])
    subprocess.run(['python','scripts/07d_generate_diffusion.py','--checkpoint',sel['best_checkpoint'],
        '--n-samples','5000','--batch-size','64','--out-tag','07d_diffusion_pool'],check=True)

### 5.filter Jüri (3-fold baseline ensemble) ile filtrele — τ=0.6 taban + rastgele N (Q2/Q3)

In [ ]:
import subprocess, json
from pathlib import Path
dmap=[('Disc space narrowing','case_1','disc_space_narrowing'),('Vertebral collapse','case_1','vertebral_collapse'),('Foraminal stenosis','case_2','foraminal_stenosis'),('Spondylolysthesis','case_2','spondylolysthesis'),('Surgical implant','case_3','surgical_implant'),('Other lesions','case_4','other_lesions')]
for nm,case,slug in dmap:
    subprocess.run(['python','scripts/07e_filter_synthetic.py','--case',case,'--class-name',nm,
        '--pool-manifest',f'outputs/07d_diffusion_pool/{slug}/manifest.csv',
        '--baseline-dir','outputs/03cv_baseline','--tau','0.6','--target-multiplier','3.0',
        '--out-tag','07f_diffusion_filtered'],check=True)
    print(json.loads(Path(f'outputs/07f_diffusion_filtered/{slug}/acceptance_report.json').read_text()))

### 5.fid Filtrelenmiş set kalite (FID) + kabul oranları

In [ ]:
import sys; sys.path.insert(0,'.')
import pandas as pd
from src.eval.fid import compute_fid, real_paths_for_class
dmap=[('Disc space narrowing','case_1','disc_space_narrowing'),('Vertebral collapse','case_1','vertebral_collapse'),('Foraminal stenosis','case_2','foraminal_stenosis'),('Spondylolysthesis','case_2','spondylolysthesis'),('Surgical implant','case_3','surgical_implant'),('Other lesions','case_4','other_lesions')]
rows=[]
for nm,case,slug in dmap:
    real=real_paths_for_class(f'outputs/02b_roi/{case}/train.csv', nm)
    fake=pd.read_csv(f'outputs/07f_diffusion_filtered/{slug}/manifest.csv')['path'].tolist()
    try: fid=round(compute_fid(real,fake,device='cuda'),2)
    except Exception as e: fid=float('nan'); print(slug,'fid err',e)
    rows.append({'class':nm,'n_real':len(real),'n_kept':len(fake),'FID_filtered':fid})
print(pd.DataFrame(rows).to_string(index=False))

### 5.cv CV Diffusion-augmented (FİLTRELENMİŞ sentetikler) [OPSİYONEL — patch-level ek]

In [ ]:
import glob, subprocess
mans=sorted(glob.glob('outputs/07f_diffusion_filtered/*/manifest.csv'))
print(mans)
subprocess.run(['python','scripts/03_train_cv.py','--config','configs/base.yaml','--cases','configs/cases.yaml',
    '--classifier','configs/classifier.yaml','--train-transform','roi_train_online','--balanced-sampler',
    '--out-tag','05cv_diffusion','--synth-manifest']+mans,check=True)
print(open('outputs/05cv_diffusion/cv_summary.md').read())

## 6. ProGAN arm (diffusion bittikten sonra) [GEREKLİ — sentetik üretimi]

### 6.0 Smoke

In [ ]:
!python scripts/05p_train_progan.py --config configs/base.yaml --progan configs/progan.yaml \
    --classes-filter "Other lesions" --fade-iters 300 --stab-iters 300 --out-tag 05p_smoke

### 6.1 ProGAN train — Disc space narrowing

In [ ]:
!python scripts/05p_train_progan.py --config configs/base.yaml --progan configs/progan.yaml \
    --classes-filter "Disc space narrowing" --out-tag 05p_progan

### 6.2 ProGAN train — Vertebral collapse

In [ ]:
!python scripts/05p_train_progan.py --config configs/base.yaml --progan configs/progan.yaml \
    --classes-filter "Vertebral collapse" --out-tag 05p_progan

### 6.3 ProGAN train — Foraminal stenosis

In [ ]:
!python scripts/05p_train_progan.py --config configs/base.yaml --progan configs/progan.yaml \
    --classes-filter "Foraminal stenosis" --out-tag 05p_progan

### 6.4 ProGAN train — Spondylolysthesis

In [ ]:
!python scripts/05p_train_progan.py --config configs/base.yaml --progan configs/progan.yaml \
    --classes-filter "Spondylolysthesis" --out-tag 05p_progan

### 6.5 ProGAN train — Surgical implant

In [ ]:
!python scripts/05p_train_progan.py --config configs/base.yaml --progan configs/progan.yaml \
    --classes-filter "Surgical implant" --out-tag 05p_progan

### 6.6 ProGAN train — Other lesions

In [ ]:
!python scripts/05p_train_progan.py --config configs/base.yaml --progan configs/progan.yaml \
    --classes-filter "Other lesions" --out-tag 05p_progan

### 6.QA ProGAN sample gridleri — ıraksama kontrolü (pool'dan ÖNCE bak)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt, cv2
for cdir in sorted(Path('outputs/05p_progan').glob('case_*')):
    for sdir in sorted(cdir.glob('*')):
        ss=sorted((sdir/'samples').glob('res128_*.png'))
        if not ss: continue
        print(f'{cdir.name}/{sdir.name}: {ss[-1].name}')
        plt.figure(figsize=(5,5)); plt.imshow(cv2.imread(str(ss[-1]),0),cmap='gray'); plt.axis('off'); plt.show()

### 6.pool Büyük havuz (5000) — depth5 checkpoint

In [ ]:
import subprocess
from pathlib import Path
dmap=[('Disc space narrowing','case_1','disc_space_narrowing'),('Vertebral collapse','case_1','vertebral_collapse'),('Foraminal stenosis','case_2','foraminal_stenosis'),('Spondylolysthesis','case_2','spondylolysthesis'),('Surgical implant','case_3','surgical_implant'),('Other lesions','case_4','other_lesions')]
for nm,case,slug in dmap:
    ck=Path(f'outputs/05p_progan/{case}/{slug}/checkpoints/depth5_res128.pt')
    if not ck.exists(): print('NO CKPT',slug); continue
    subprocess.run(['python','scripts/07p_generate_progan.py','--checkpoint',str(ck),
        '--n-samples','5000','--batch-size','64','--out-tag','07p_progan_pool'],check=True)

### 6.filter Jüri ile filtrele

In [ ]:
import subprocess, json
from pathlib import Path
dmap=[('Disc space narrowing','case_1','disc_space_narrowing'),('Vertebral collapse','case_1','vertebral_collapse'),('Foraminal stenosis','case_2','foraminal_stenosis'),('Spondylolysthesis','case_2','spondylolysthesis'),('Surgical implant','case_3','surgical_implant'),('Other lesions','case_4','other_lesions')]
for nm,case,slug in dmap:
    pool=f'outputs/07p_progan_pool/{slug}/manifest.csv'
    if not Path(pool).exists(): print('NO POOL',slug); continue
    subprocess.run(['python','scripts/07e_filter_synthetic.py','--case',case,'--class-name',nm,
        '--pool-manifest',pool,'--baseline-dir','outputs/03cv_baseline','--tau','0.6',
        '--target-multiplier','3.0','--out-tag','07f_progan_filtered'],check=True)
    print(json.loads(Path(f'outputs/07f_progan_filtered/{slug}/acceptance_report.json').read_text()))

### 6.cv CV ProGAN-augmented (filtrelenmiş) [OPSİYONEL — patch-level ek]

In [ ]:
import glob, subprocess
mans=sorted(glob.glob('outputs/07f_progan_filtered/*/manifest.csv'))
print(mans)
subprocess.run(['python','scripts/03_train_cv.py','--config','configs/base.yaml','--cases','configs/cases.yaml',
    '--classifier','configs/classifier.yaml','--train-transform','roi_train_online','--balanced-sampler',
    '--out-tag','06cv_progan','--synth-manifest']+mans,check=True)
print(open('outputs/06cv_progan/cv_summary.md').read())

## 7. Final karşılaştırma tablosu [OPSİYONEL — patch-level ek]

In [ ]:
import json, pandas as pd
from pathlib import Path
conds=[('03cv_baseline','baseline'),('04cv_traditional','traditional'),('05cv_diffusion','diffusion'),('06cv_progan','progan')]
recs=[]
for tag,lbl in conds:
    f=Path(f'outputs/{tag}/cv_summary.json')
    if not f.exists(): continue
    for s in json.loads(f.read_text()):
        recs.append({'case':s['case'],'condition':lbl,'macroF1':f"{s['test_macro_f1_mean']:.4f}±{s['test_macro_f1_std']:.4f}"})
df=pd.DataFrame(recs)
if len(df): print(df.pivot(index='case',columns='condition',values='macroF1').to_string())

## 8. Tez raporu üret (figürler + LaTeX/MD tablolar + insights) [GEREKLİ — en son]

Tüm mevcut koşulları otomatik tarar; ProGAN bitmemişse onsuz üretir. CM/ROC/PR için best.pth'leri yükleyip test setinde yeniden çıkarım yapar (GPU).

In [ ]:
!python scripts/06_generate_report.py --config configs/base.yaml --cases configs/cases.yaml
print('--- report.md ---'); print(open('outputs/06_reports/report.md').read())

### 8.1 Anahtar figürleri göster

In [ ]:
from pathlib import Path
from IPython.display import Image, display
for name in ['macro_f1_bars','win_heatmap','fold_variance_boxplots']:
    p=Path(f'outputs/06_reports/figures/{name}.png')
    if p.exists(): print(name); display(Image(str(p)))
for p in sorted(Path('outputs/06_reports/figures').glob('perclass_f1_*.png')): display(Image(str(p)))
for p in sorted(Path('outputs/06_reports/figures').glob('montage_*.png')): display(Image(str(p)))

### 8.2 Tabloları göster

In [ ]:
from IPython.display import Markdown, display
for t in ['macro_f1','per_class_f1','synthetic_quality']:
    p=f'outputs/06_reports/tables/{t}.md'
    import os
    if os.path.exists(p): display(Markdown(open(p).read()))

## 9. PATIENT-LEVEL pipeline (hocanın istediği gerçek test — GT box test'te YOK) [HEADLINE — hocanın istediği gerçek test]

Eğitim 128 native pencere (pozitif + **hard-negative arka plan** + NF) ile; test tam görüntüde stride-96 kayan pencere, kutu kullanılmadan. Patient-level multi-label macro-F1.

**Beklenti:** mutlak F1 patch-level'a göre ciddi düşer (dürüst); önemli olan baseline < traditional ≤ diffusion/progan GAP'i.

### 9.1 128 native pencere patch'lerini üret (pos + hard-neg + nf)

In [ ]:
!python scripts/02d_build_window_patches.py --config configs/base.yaml --cases configs/cases.yaml
!cat outputs/02d_windows/summary.md

### 9.2 Virtual patients (diffusion + progan filtrelenmiş sentetikler)

In [ ]:
!python scripts/build_virtual_patients.py --config configs/base.yaml --filtered-tag 07f_diffusion_filtered --out-tag virtual_patients_diffusion
!python scripts/build_virtual_patients.py --config configs/base.yaml --filtered-tag 07f_progan_filtered --out-tag virtual_patients_progan

### 9.3 Train baseline_win (aug yok)

In [ ]:
!python scripts/03w_train_window.py --config configs/base.yaml --cases configs/cases.yaml \
    --classifier configs/classifier.yaml --train-transform baseline_no_aug --out-tag baseline_win

### 9.4 Train traditional_win (online aug + balanced sampler)

In [ ]:
!python scripts/03w_train_window.py --config configs/base.yaml --cases configs/cases.yaml \
    --classifier configs/classifier.yaml --train-transform roi_train_online --balanced-sampler --out-tag traditional_win

### 9.5 Train diffusion_win (+ virtual-patient sentetikler)

In [ ]:
import glob
mans=sorted(glob.glob('outputs/virtual_patients_diffusion/*/manifest.csv'))
print(mans)
import subprocess
subprocess.run(['python','scripts/03w_train_window.py','--config','configs/base.yaml','--cases','configs/cases.yaml',
  '--classifier','configs/classifier.yaml','--train-transform','roi_train_online','--balanced-sampler',
  '--out-tag','diffusion_win','--synth-manifest']+mans,check=True)

### 9.6 Train progan_win (+ virtual-patient sentetikler)

In [ ]:
import glob, subprocess
mans=sorted(glob.glob('outputs/virtual_patients_progan/*/manifest.csv'))
print(mans)
subprocess.run(['python','scripts/03w_train_window.py','--config','configs/base.yaml','--cases','configs/cases.yaml',
  '--classifier','configs/classifier.yaml','--train-transform','roi_train_online','--balanced-sampler',
  '--out-tag','progan_win','--synth-manifest']+mans,check=True)

### 9.7 Calibrate (val) + Test (resmi test, full image) — her koşul

Her koşul için önce eşikleri internal-val full görüntülerinde kalibre et, sonra resmi test'te koş. Test kutu KULLANMAZ.

In [ ]:
import subprocess
for cond in ['baseline_win','traditional_win','diffusion_win','progan_win']:
    print('===',cond,'CALIBRATE ==='); 
    subprocess.run(['python','scripts/08_patient_level_inference.py','--cond-tag',cond,'--mode','calibrate','--stride','96'],check=True)
    print('===',cond,'TEST ===')
    subprocess.run(['python','scripts/08_patient_level_inference.py','--cond-tag',cond,'--mode','test','--stride','96'],check=True)
    print(open(f'outputs/{cond.replace("_win","_patient")}/per_disease.md').read())

### 9.8 Patient-level karşılaştırma tablosu

In [ ]:
import json, pandas as pd
from pathlib import Path
rows=[]
for cond in ['baseline','traditional','diffusion','progan']:
    f=Path(f'outputs/{cond}_patient/metrics.json')
    if not f.exists(): continue
    m=json.loads(f.read_text())
    row={'condition':cond,'patient_macro_F1':round(m['patient_macro_f1'],4)}
    for d,v in m['per_disease'].items(): row[d[:14]]=round(v['f1'],3)
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))